# UD Dependency Distances Across Languages and Registers

## Exploring Dependency Distance Minimization (DDM) in Universal Dependencies

This notebook demonstrates an analysis of **dependency-distance distributions** across 18 Universal Dependencies v2.18 treebanks, covering:

- **Matched spoken/written register pairs** in the same language (Slovenian SST/SSJ, French Rhapsodie/GSD, English ESLSpok/EWT/GUM)
- **Typologically diverse treebanks** spanning multiple language families (Indo-European, Turkic, Afro-Asiatic, Japonic, Koreanic, Uralic, Sino-Tibetan, Creole)
- **Grambank typological features** (morphosyntax, word order, case/agreement patterns) joined per language

**Key outputs:**
- Per-arc **raw and length-normalized dependency distances** (|dependent_position - head_position|)
- **Mean distance by sentence**, language, and register
- **Head-finality ratio** (fraction of arcs where head follows dependent)
- Comparison across register pairs to investigate **Dependency Distance Minimization** and how it varies by language typology

This is a demo on a small curated subset (~10 sentences). For the full analysis, use the production dataset (33,030 sentences across 18 treebanks).

## 1. Install Dependencies

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Core packages: install locally to match Colab, skip on Colab to avoid ABI corruption
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0')

print("✓ Dependencies installed")

## 2. Imports

In [ ]:
import json
import sys
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print(f"Python {sys.version.split()[0]}")
print(f"NumPy {np.__version__}")
print(f"Pandas {pd.__version__}")

## 3. Data Loading Function

Load data from GitHub URL with fallback to local file. This pattern works in both Colab and local Jupyter environments.

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-544c17-language-minimizes-dependency-distance/main/round-1/dataset-1/demo/mini_demo_data.json"

def load_data():
    """Load mini demo data from GitHub URL with local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        pass
    
    if Path("mini_demo_data.json").exists():
        with open("mini_demo_data.json") as f:
            return json.load(f)
    
    raise FileNotFoundError("Could not load mini_demo_data.json from GitHub or local path")

print("✓ Data loader defined")

## 4. Load Data

In [ ]:
data = load_data()

metadata = data['metadata']
examples = data['datasets'][0]['examples']

print(f"✓ Loaded {len(examples)} examples")
print(f"  Source: {metadata['source'][:80]}...")
print(f"  Examples with Grambank typology: {metadata['n_examples_with_grambank_typology']}")

## 5. Configuration

**All parameters are set to MINIMUM values for a quick demo.** Scale up by modifying these variables for full analysis.

In [ ]:
# === CONFIG: All tunable parameters ===

# Number of examples to process (set to min for demo; increase for larger runs)
N_EXAMPLES = 10  # Set to len(examples) for full dataset

# Thresholds for dependency distance analysis
MIN_SENTENCE_LENGTH = 3  # Ignore very short sentences
MIN_NORMALIZED_DISTANCE = 0.0  # Minimum normalized DDM for inclusion in stats
MAX_NORMALIZED_DISTANCE = 1.0  # Maximum normalized DDM

# Visualization parameters
PLOT_DPI = 100
PLOT_FIGSIZE = (12, 5)

# ===

print(f"Config: Processing {N_EXAMPLES}/{len(examples)} examples")
print(f"Sentence length: {MIN_SENTENCE_LENGTH}-inf tokens")

## 6. Parse and Extract Features

Extract dependency distances, normalized distances, and metadata from each sentence. Parse JSON fields from the dataset.

In [ ]:
# Parse examples and extract dependency distance statistics
rows = []

for i, ex in enumerate(examples[:N_EXAMPLES]):
    # Parse nested JSON fields
    input_obj = json.loads(ex['input'])
    output_obj = json.loads(ex['output'])
    
    # Extract distances
    distances = output_obj.get('dependency_distances', [])
    normalized = output_obj.get('normalized_distances', [])
    
    # Build row for analysis
    row = {
        'example_id': i,
        'treebank_id': ex['metadata_treebank_id'],
        'language': ex['metadata_language'],
        'iso639_3': ex['metadata_iso639_3'],
        'language_family': ex['metadata_language_family'],
        'register': ex['metadata_register'],
        'sentence_length': ex['metadata_sentence_length'],
        'num_arcs': ex['metadata_num_arcs'],
        'mean_distance': ex['metadata_mean_dependency_distance'],
        'mean_normalized_distance': ex['metadata_mean_normalized_distance'],
        'head_finality_ratio': ex['metadata_head_finality_ratio'],
        'num_distances': len(distances),
        'text': input_obj.get('text', ''),
    }
    
    # Only include if meets minimum requirements
    if row['sentence_length'] >= MIN_SENTENCE_LENGTH:
        rows.append(row)

df = pd.DataFrame(rows)
print(f"✓ Extracted {len(df)} valid examples (sentence_length >= {MIN_SENTENCE_LENGTH})")
print(f"\nDataFrame shape: {df.shape}")
print(f"Languages: {df['language'].nunique()}")
print(f"Registers: {df['register'].unique()}")

## 7. Summary Statistics by Language and Register

Compute mean dependency distances, normalized distances, and head-finality ratios grouped by language and register.

In [ ]:
# Group by language and register
summary = df.groupby(['language', 'register']).agg({
    'mean_distance': ['mean', 'std', 'count'],
    'mean_normalized_distance': ['mean', 'std'],
    'head_finality_ratio': ['mean', 'std'],
    'sentence_length': ['mean', 'min', 'max'],
}).round(4)

print("Summary Statistics by Language and Register:")
print(summary)

# Also show a simpler view
simple_summary = df.groupby(['language', 'register']).agg({
    'mean_distance': 'mean',
    'mean_normalized_distance': 'mean',
    'head_finality_ratio': 'mean',
    'example_id': 'count'
}).rename(columns={'example_id': 'n_sentences'}).round(4)

print("\nSimplified Summary (Mean values):")
print(simple_summary)

## 8. Dependency Distance Distributions

Compare raw and normalized dependency distances across languages and registers. Check for evidence of Dependency Distance Minimization.

In [ ]:
# Compare mean distances across language families
by_family = df.groupby('language_family').agg({
    'mean_distance': ['mean', 'std', 'count'],
    'mean_normalized_distance': ['mean', 'std'],
    'head_finality_ratio': 'mean',
}).round(4)

print("Dependency Distances by Language Family:")
print(by_family)

# Simpler view
family_summary = df.groupby('language_family').agg({
    'mean_distance': 'mean',
    'mean_normalized_distance': 'mean',
    'head_finality_ratio': 'mean',
    'example_id': 'count'
}).rename(columns={'example_id': 'n_examples'}).sort_values('mean_normalized_distance', ascending=False).round(4)

print("\nSimplified by Language Family (sorted by normalized distance):")
print(family_summary)

## 9. Head-Finality Analysis

Analyze the **head-finality ratio** — the fraction of arcs where the head appears AFTER the dependent. This is a key typological feature associated with word order.

In [ ]:
# Analyze head-finality (fraction of arcs where head follows dependent)
print("Head-Finality Analysis:")
print("(Higher ratio = more head-final language, typical of SOV; lower = more head-initial, typical of SVO)\n")

hf_stats = df.groupby('language').agg({
    'head_finality_ratio': ['mean', 'std'],
    'example_id': 'count'
}).rename(columns={'example_id': 'n_examples'}).round(4)

hf_stats.columns = ['_'.join(col).strip() for col in hf_stats.columns.values]
hf_stats = hf_stats.sort_values('head_finality_ratio_mean', ascending=False)

print(hf_stats)

# Summary stats
print(f"\nMean head-finality across all examples: {df['head_finality_ratio'].mean():.4f}")
print(f"Std dev: {df['head_finality_ratio'].std():.4f}")
print(f"Range: [{df['head_finality_ratio'].min():.4f}, {df['head_finality_ratio'].max():.4f}]")

## 10. Register Comparison (Spoken vs Written)

For languages with both spoken and written data, compare dependency distance metrics to test the hypothesis that **spoken language minimizes dependency distances more than written language**.

In [ ]:
# Find languages with multiple registers
multi_register_langs = df.groupby('language')['register'].nunique()
multi_register_langs = multi_register_langs[multi_register_langs > 1].index.tolist()

print(f"Languages with multiple registers: {multi_register_langs}\n")

if multi_register_langs:
    register_comparisons = []
    for lang in multi_register_langs:
        lang_data = df[df['language'] == lang]
        for register in lang_data['register'].unique():
            subset = lang_data[lang_data['register'] == register]
            register_comparisons.append({
                'language': lang,
                'register': register,
                'n_examples': len(subset),
                'mean_distance': subset['mean_distance'].mean(),
                'mean_normalized_distance': subset['mean_normalized_distance'].mean(),
                'head_finality': subset['head_finality_ratio'].mean(),
            })
    
    register_df = pd.DataFrame(register_comparisons)
    print("Register Comparison (Spoken vs Written):")
    print(register_df.sort_values(['language', 'register']).round(4))
    
    # Compute differences
    print("\nDifferences (spoken - written):")
    for lang in multi_register_langs:
        subset = register_df[register_df['language'] == lang]
        if len(subset) > 1:
            spoken = subset[subset['register'] == 'spoken']
            written = subset[subset['register'] == 'written']
            if not spoken.empty and not written.empty:
                diff_dist = spoken['mean_normalized_distance'].values[0] - written['mean_normalized_distance'].values[0]
                print(f"  {lang}: {diff_dist:+.4f} (spoken - written normalized distance)")
else:
    print("No languages with multiple registers in this sample.")

## 11. Visualizations

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=PLOT_FIGSIZE, dpi=PLOT_DPI)
fig.suptitle('Dependency Distance Minimization Across Languages', fontsize=14, fontweight='bold')

# Plot 1: Mean normalized distance by language
ax = axes[0, 0]
lang_means = df.groupby('language')['mean_normalized_distance'].mean().sort_values()
ax.barh(range(len(lang_means)), lang_means.values, color='steelblue')
ax.set_yticks(range(len(lang_means)))
ax.set_yticklabels(lang_means.index, fontsize=9)
ax.set_xlabel('Mean Normalized Distance')
ax.set_title('(a) Dependency Distance by Language')
ax.grid(axis='x', alpha=0.3)

# Plot 2: Head-finality ratio by language
ax = axes[0, 1]
hf_means = df.groupby('language')['head_finality_ratio'].mean().sort_values()
ax.barh(range(len(hf_means)), hf_means.values, color='coral')
ax.set_yticks(range(len(hf_means)))
ax.set_yticklabels(hf_means.index, fontsize=9)
ax.set_xlabel('Head-Finality Ratio')
ax.set_title('(b) Head-Finality by Language')
ax.set_xlim([0, 1])
ax.grid(axis='x', alpha=0.3)

# Plot 3: Sentence length distribution
ax = axes[1, 0]
ax.hist(df['sentence_length'], bins=15, edgecolor='black', alpha=0.7, color='green')
ax.set_xlabel('Sentence Length (tokens)')
ax.set_ylabel('Frequency')
ax.set_title('(c) Sentence Length Distribution')
ax.grid(axis='y', alpha=0.3)

# Plot 4: Scatter plot: sentence length vs mean normalized distance
ax = axes[1, 1]
registers = df['register'].unique()
colors = {'spoken': 'red', 'written': 'blue', 'mixed': 'purple'}
for reg in registers:
    subset = df[df['register'] == reg]
    ax.scatter(subset['sentence_length'], subset['mean_normalized_distance'], 
              label=reg, alpha=0.6, s=60, color=colors.get(reg, 'gray'))
ax.set_xlabel('Sentence Length (tokens)')
ax.set_ylabel('Mean Normalized Distance')
ax.set_title('(d) Sentence Length vs DDM')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('ddm_analysis.png', dpi=PLOT_DPI, bbox_inches='tight')
plt.show()

print("✓ Visualization saved as ddm_analysis.png")

## 12. Summary and Key Findings

In [ ]:
print("="*70)
print("SUMMARY: Dependency Distance Minimization Analysis")
print("="*70)

print(f"\n✓ Dataset: {len(df)} sentences from {df['language'].nunique()} languages")
print(f"✓ Language families: {df['language_family'].nunique()}")
print(f"✓ Registers: {', '.join(df['register'].unique())}")

print(f"\n📊 Dependency Distance Metrics:")
print(f"  Mean raw distance: {df['mean_distance'].mean():.4f} ± {df['mean_distance'].std():.4f} tokens")
print(f"  Mean normalized distance: {df['mean_normalized_distance'].mean():.4f} ± {df['mean_normalized_distance'].std():.4f}")
print(f"  Mean head-finality ratio: {df['head_finality_ratio'].mean():.4f}")

print(f"\n📏 Sentence Properties:")
print(f"  Avg sentence length: {df['sentence_length'].mean():.2f} tokens")
print(f"  Avg arcs per sentence: {df['num_arcs'].mean():.2f}")

print(f"\n🌍 Language Families (ranked by DDM):")
top_families = family_summary[['mean_normalized_distance', 'mean_distance']].head()
for i, (fam, row) in enumerate(top_families.iterrows(), 1):
    print(f"  {i}. {fam:20s} — norm_dist={row['mean_normalized_distance']:.4f}, raw_dist={row['mean_distance']:.4f}")

print(f"\n💡 Next Steps:")
print(f"  • Increase N_EXAMPLES in config to analyze full dataset (33,030 sentences)")
print(f"  • Test hypothesis: Does spoken language show lower DDM (more distance minimization)?")
print(f"  • Fit extreme-value models (POT, GPD) to dependency distance tails")
print(f"  • Use Grambank features as typological covariates in mixed-effects models")
print(f"  • Analyze within-language register pairs more carefully")

print("\n" + "="*70)

## Appendix: Data Sample

Show a few raw examples from the dataset to understand the structure.

In [ ]:
print("Sample Examples from Dataset:\n")

for i in range(min(3, len(df))):
    row = df.iloc[i]
    print(f"Example {i+1}: {row['language']} ({row['register']})")
    print(f"  Text: {row['text'][:80]}...")
    print(f"  Sentence length: {row['sentence_length']} tokens")
    print(f"  Num arcs: {row['num_arcs']}")
    print(f"  Mean distance: {row['mean_distance']:.4f} (normalized: {row['mean_normalized_distance']:.4f})")
    print(f"  Head finality ratio: {row['head_finality_ratio']:.4f}")
    print()